# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [4]:
# imports
import os
import httpx
import json
from IPython.display import Markdown, display, update_display
from dotenv import load_dotenv
from openai import OpenAI

In [20]:
# constants

MODEL_GPT = 'gpt-5-mini'
MODEL_GEMINI = 'qwen3.6-27b'

In [3]:
# set up environment
load_dotenv(override=True)

BASE_URL = os.getenv("LITELLM_BASE_URL")
LLM_API_KEY = os.getenv("LITELLM_API_KEY")

openai = OpenAI(
    base_url=BASE_URL,
    api_key=LLM_API_KEY,
    http_client=httpx.Client(verify=False, timeout=60.0)
)

In [9]:
def system_user_interaction(model, system_prompt, user_prompt):
    stream = openai.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        stream=True
    )

    response = ""

    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [10]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [11]:
# Get gpt-4o-mini to answer, with streaming
system_prompt = """
Please answer in bahasa melayu and answer like explain to the kids 5 years old.
"""

system_user_interaction(MODEL_GPT, system_prompt, question)

Bayangkan kamu ada sekotak buku. Setiap buku ada nama penulis. Kita mahu bagi nama-nama penulis itu satu per satu, tapi kita tidak mahu nama yang kosong dan kita mahu setiap nama hanya sekali saja.

Baris kod itu buat tiga benda:

1. "{book.get('author') for book in books if book.get('author')}"
   - Ini macam mengutip semua nama penulis dari setiap buku.
   - Ia simpan dalam satu kotak khas yang panggil "set" — kotak ini cuma simpan satu salinan bagi setiap nama (jadi ia buang yang sama).
   - Ia juga tolakkan nama yang kosong atau tiada (kerana ada "if book.get('author')").

2. "yield from <set>"
   - Ini macam kamu berdiri dan bagi keluar satu persatu nama dari kotak tadi kepada orang lain.
   - Jadi fungsi itu "mengeluarkan" setiap nama penulis dari set tadi.

Kenapa buat macam ini?
- Supaya dapatkan hanya penulis yang wujud dan buang yang kosong.
- Supaya setiap nama hanya muncul sekali (tak ulang).

Perkara yang perlu hati-hati (kekurangan):
- Set tidak ikut susunan asal. Nama yang keluar boleh jadi dalam susunan rawak.
- Kod itu panggil book.get('author') dua kali — sedikit tak efisien.
- Membina set ambil memori — kalau buku sangat banyak, boleh pakai cara lain.

Cara yang lebih baik jika mahu kekalkan susunan asal dan elak panggilan berganda:
- Simpan penulis dalam pembolehubah sementara dan guna "seen" untuk pastikan unik, kemudian yield satu persatu. Contoh ringkas idea:
  - for book in books:
      author = book.get('author')
      if author and author not in seen:
          seen.add(author)
          yield author

Itu saja — beri nama penulis yang unik, satu per satu.

In [13]:
# Get Llama 3.2 to answer

system_prompt = """
Please answer in bahasa melayu and answer like explain to the kids 5 years old.
"""

system_user_interaction(MODEL_GEMINI, system_prompt, question)

Hai adik! Mari abang/kakak ceritakan satu cerita pasal kod komputer yang comel ni. 

Bayangkan adik ada satu **bakul mainan yang penuh dengan buku cerita** (`books`). 

Kod komputer ni macam seorang pembantu kecil yang sangat rajin. Jom tengok apa yang dia buat:

### 1. Dia cari nama "Penulis" (`book.get("author")`)
Pembantu kecil ni akan buka buku tu satu-satu untuk cari: *"Siapa nama pakcik atau makcik yang tulis buku ni ya?"* 

### 2. Dia pilih buku yang ada nama sahaja (`if book.get("author")`)
Kalau buku tu tak ada nama penulis, pembantu ni akan ketepikan. Dia cuma ambil buku yang **ada nama penulis** sahaja.

### 3. Dia masukkan dalam "Beg Ajaib" (`{ ... }`)
Nama-nama penulis tadi dimasukkan ke dalam satu **beg ajaib** (simbol `{ }` dalam kod tu). 
Kenapa beg ni ajaib? Sebab kalau ada dua buku yang ditulis oleh orang yang sama (contohnya: "Uncle Bob"), beg ajaib ni akan **hilangkan nama yang sama**. Jadi, nama "Uncle Bob" cuma ada satu saja dalam beg tu. Tak ada yang bertindan!

### 4. Dia bagi kat kita satu-satu (`yield from`)
Akhir sekali, pembantu comel ni sebut `yield from`. Maksudnya, dia akan seluk tangan dalam beg ajaib tu, pastu dia **hulurkan nama-nama penulis tu kepada kita satu demi satu dengan sopan**. 
"Nah nama pertama... nah nama kedua..." sampai habis!

---

### Kesimpulannya:
Kod ni sebenarnya tolong kita **kumpul semua nama penulis buku, buang nama yang sama, dan bagi kat kita nama-nama tu satu-satu untuk kita baca.** 

Hebat dan rajin kan pembantu kecil ni? Hehe.

In [18]:
system_prompt = """
Please answer this question in a simple way and concise especially to the people that always think they are right. 
Please answer in bahasa melayu
"""

user_prompt = """
Siapa it Dajjal dalam islam?
"""

In [19]:
system_user_interaction(
    MODEL_GPT, 
    system_prompt=system_prompt, 
    user_prompt=user_prompt
    )

Dajjal ialah tokoh penipu besar dalam ajaran Islam — “khalifah palsu” atau “mesias palsu” yang muncul menjelang hari kiamat untuk menguji keimanan manusia.

Poin ringkas:
- Sifat: menurut hadis, dia akan buta sebelah mata (sering digambarkan bermata satu) dan membawa fitnah besar.  
- Tindakannya: membuat mukjizat palsu, menyesatkan ramai orang dengan tipu daya kekayaan, makanan dan kuasa.  
- Tempat dia tak boleh masuk: Makkah dan Madinah dilindungi malaikat.  
- Pengakhiran: Nabi Isa (Yesus) akan turun kembali dan membunuh Dajjal.  
- Cara bertahan: kuatkan iman, pelihara syahadah, baca Surah Al-Kahf, berzikir dan ikut ajaran Nabi.  
- Sumber: maklumat ini berdasarkan hadis sahih (Bukhari & Muslim) dan ajaran ulama.

Untuk yang selalu rasa betul: dengar dulu sumbernya — ilmu tentang akhir zaman ada dalilnya, bukan tekaan. Jangan cepat menolak atau mengaku lebih tahu tanpa bukti.

In [21]:
system_user_interaction(
    MODEL_GEMINI,
    system_prompt=system_prompt,
    user_prompt=user_prompt
)

Dajjal ialah makhluk **satu mata** yang akan muncul sebelum hari kiamat. Dia bukan manusia biasa, malah disebut sebagai "setan" kerana akan menyesatkan manusia dengan meniru mukjizat (menyembuhkan penyakit, menghidupkan mayat) tetapi sebenarnya itu adalah tipu muslihat syaitan.

**Ingat:** Dia tidak pernah benar, hanya menipu. Percayalah hanya pada ajaran Allah dan Rasul-Nya.